# Modular-addition verifier sandbox

This notebook trains a tiny transformer from scratch on part of an addition table modulo a prime. It then applies verifier operating points `(alpha, beta)` and compares measured accepted accuracy with

$$a' = \frac{\alpha a}{\alpha a + \beta(1-a)}.$$

The smoke config has about 14,000 parameters. It downloads no model and launches no cloud job.

In [ ]:
from pathlib import Path
import os
import sys

def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Open this notebook from inside the cloned repository.")

REPOSITORY_ROOT = find_repository_root(Path.cwd())
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
source_path = str(SOURCE_DIRECTORY)
if source_path not in sys.path:
    sys.path.insert(0, source_path)
os.chdir(REPOSITORY_ROOT)
print("Repository:", REPOSITORY_ROOT)
print("Local package source:", SOURCE_DIRECTORY)

In [ ]:
from verifier_bottleneck.experiments.modular_addition import (
    describe_experiment,
    load_experiment_config,
)

CONFIG_PATH = Path("configs/sandbox/modular_addition_smoke.yaml")
config = load_experiment_config(CONFIG_PATH)
describe_experiment(config)

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Review the dry-run description above. The default smoke run is intentionally tiny. Set `RUN_EXPERIMENT = True` only when you are ready to use the currently selected notebook runtime.

In [ ]:
from verifier_bottleneck.experiments.modular_addition import run_from_path

RUN_EXPERIMENT = False
result = None

if RUN_EXPERIMENT:
    result = run_from_path(CONFIG_PATH)
    print("Run ID:", result["run_id"])
    print("Record:", result["record_path"])
    print("Summary:", result["summary_path"])
else:
    print("Set RUN_EXPERIMENT = True to start the smoke run.")

In [ ]:
if result is not None:
    import matplotlib.pyplot as plt

    from verifier_bottleneck.experiment_tracking import register_artifact

    trajectory = result["metrics"]["trajectory"]
    steps = [row["step"] for row in trajectory]
    train_accuracy = [row["train_accuracy"] for row in trajectory]
    test_accuracy = [row["test_accuracy"] for row in trajectory]

    final_measurements = trajectory[-1]["verifier_measurements"]
    labels = [f"α={row['alpha']}, β={row['beta']}" for row in final_measurements]
    predicted = [row["predicted_accepted_accuracy"] for row in final_measurements]
    empirical = [row["empirical_accepted_accuracy"] for row in final_measurements]

    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(steps, train_accuracy, label="train")
    axes[0].plot(steps, test_accuracy, label="test")
    axes[0].set(xlabel="step", ylabel="accuracy", ylim=(0, 1.02))
    axes[0].legend()

    positions = list(range(len(labels)))
    axes[1].plot(positions, predicted, "o-", label="predicted a′")
    axes[1].plot(positions, empirical, "x--", label="empirical a′")
    axes[1].set_xticks(positions, labels, rotation=30, ha="right")
    axes[1].set(ylabel="accepted accuracy", ylim=(0, 1.02))
    axes[1].legend()

    figure.tight_layout()
    plot_path = Path(result["run_directory"]) / "accuracy-and-verifier.png"
    figure.savefig(plot_path, dpi=150)
    register_artifact(
        Path(result["record_path"]),
        plot_path,
        kind="plot",
        description="Training/test accuracy and predicted versus empirical verifier accuracy.",
    )
    print("Plot:", plot_path)
    plt.show()

After the smoke run works, switch `CONFIG_PATH` to `configs/sandbox/modular_addition_paper.yaml` only after reviewing its 100,000-step budget. The paper-scale config is not a smoke test.